# Eulerian Video Magnification

<p>Eulerian video magnification reveals temporal variations in videos that are difficult or impossible to see with the naked eye and display them in an indicative manner.
</p>
<p>We are going to create a system which takes in an input video and outputs a video that is motion magnified. The system first decomposes the input video sequence into different
spatial frequency bands, and applies the same temporal filter to all bands. The filtered spatial bands are then amplified by a given factor $\alpha$,
added back to the original signal, and collapsed to generate the output video.</p> 


The Algorithm we use is derived from MIT CSAIL's paper, ["Eulerian Video Magnification for Revealing Subtle Changes in the World"](http://people.csail.mit.edu/mrub/papers/vidmag.pdf). I have implemented their paper using Python.


##Overview of the Eulerian video magnification framework


![](https://github.com/joeljose/assets/blob/master/EVM/EVM_flow.png?raw=True)



##There are 5 steps in the algorithm pipeline:
1) Loading the video</br> 
2) Spatial decomposition into laplacian pyramids</br>
3) Temporal filtering to extract motion information, and adding that back to the original signal</br>
4) Reconstruction </br>
5) Saving to output video</br>
We will first create helper functions for each of these steps. And finally put them altogether to get our motion magnification method.


## Importing all the essential modules:

In [ ]:
import math
import os

import cv2
import numpy as np
import scipy.fftpack
from matplotlib import pyplot
import requests

## helper functions for loading and saving videos:

In [ ]:
# YIQ/NTSC color space conversion matrices (matches MATLAB rgb2ntsc/ntsc2rgb)
_RGB_TO_YIQ = np.array([
    [0.299, 0.587, 0.114],
    [0.596, -0.274, -0.322],
    [0.211, -0.523, 0.312],
], dtype=np.float32)

_YIQ_TO_RGB = np.linalg.inv(_RGB_TO_YIQ).astype(np.float32)


def rgb_to_yiq(frame):
    """Convert an RGB float frame to YIQ color space."""
    return frame @ _RGB_TO_YIQ.T


def yiq_to_rgb(frame):
    """Convert a YIQ float frame to RGB color space."""
    return frame @ _YIQ_TO_RGB.T


def load_video(video_filename):
    """Load a video file and return (YIQ float32 numpy array, fps).

    The returned array has shape (num_frames, height, width, 3) in YIQ
    color space with Y in [0, 1].
    """
    video_filename = str(video_filename)
    print(f"Loading {video_filename}")
    if not os.path.isfile(video_filename):
        raise FileNotFoundError(f"File Not Found: {video_filename}")
    cap = cv2.VideoCapture(video_filename)
    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))

        frames = np.zeros((frame_count, height, width, 3), dtype='uint8')
        i = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frames[i] = frame
            i += 1
    finally:
        cap.release()

    # Trim to actual frame count, convert BGR->RGB->float->YIQ
    frames = frames[:i]
    rgb = frames[:, :, :, ::-1].astype(np.float32) / 255.0
    del frames
    yiq = rgb @ _RGB_TO_YIQ.T
    del rgb
    return yiq, fps


def save_video(video_yiq, fps, save_filename='output.avi'):
    """Save a YIQ float video array to an AVI file with MJPG codec."""
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    h, w = video_yiq.shape[1], video_yiq.shape[2]
    writer = cv2.VideoWriter(save_filename, fourcc, fps, (w, h), True)
    try:
        for i in range(video_yiq.shape[0]):
            rgb = yiq_to_rgb(video_yiq[i])
            bgr = np.clip(rgb[:, :, ::-1] * 255, 0, 255).astype(np.uint8)
            writer.write(bgr)
    finally:
        writer.release()
    print(f"Output saved to {save_filename}")

##Helper functions for Spatial Decomposition(Laplacian pyramids), Temporal filtering, and Reconstruction

In [ ]:
def create_gaussian_image_pyramid(image, pyramid_levels):
    """Create a gaussian image pyramid for the input image."""
    gauss_copy = np.ndarray(shape=image.shape, dtype="float")
    gauss_copy[:] = image
    img_pyramid = [gauss_copy]
    for pyramid_level in range(1, pyramid_levels):
        gauss_copy = cv2.pyrDown(gauss_copy)
        img_pyramid.append(gauss_copy)

    return img_pyramid


def create_laplacian_image_pyramid(image, pyramid_levels):
    """Create a laplacian image pyramid for the input image."""
    gauss_pyramid = create_gaussian_image_pyramid(image, pyramid_levels)
    laplacian_pyramid = []
    for i in range(pyramid_levels - 1):
        laplacian_pyramid.append(
            gauss_pyramid[i] - cv2.pyrUp(gauss_pyramid[i + 1])
        )

    laplacian_pyramid.append(gauss_pyramid[-1])
    return laplacian_pyramid


def create_laplacian_video_pyramid(video, pyramid_levels):
    """Create a laplacian video pyramid for the input video."""
    vid_pyramid = []
    for frame_number, frame in enumerate(video):
        frame_pyramid = create_laplacian_image_pyramid(
            frame, pyramid_levels
        )

        for pyramid_level, pyramid_sub_frame in enumerate(frame_pyramid):
            if frame_number == 0:
                vid_pyramid.append(np.zeros(
                    (video.shape[0],
                     pyramid_sub_frame.shape[0],
                     pyramid_sub_frame.shape[1], 3),
                    dtype="float"
                ))

            vid_pyramid[pyramid_level][frame_number] = pyramid_sub_frame

    return vid_pyramid


def temporal_bandpass_filter(data, fps, freq_min=0.833, freq_max=1):
    """Apply ideal bandpass filter along the time axis.

    Matches the reference MATLAB implementation (ideal_bandpassing.m):
    - One-sided frequency mask (positive frequencies only)
    - Returns real part of IFFT (not absolute value)
    - No amplification applied (done separately per level)
    """
    n = data.shape[0]
    freqs = np.arange(n) / n * fps
    mask = ((freqs > freq_min) & (freqs < freq_max)).astype(np.float64)
    mask = mask.reshape([n] + [1] * (data.ndim - 1))

    fft = scipy.fftpack.fft(data, axis=0)
    fft *= mask

    return np.real(scipy.fftpack.ifft(fft, axis=0)).astype(np.float32)


def collapse_laplacian_pyramid(image_pyramid):
    """Reconstruct an image from a laplacian pyramid."""
    img = image_pyramid[-1]
    for i in range(len(image_pyramid) - 2, -1, -1):
        img = cv2.pyrUp(img) + image_pyramid[i]
    return img


def collapse_laplacian_video_pyramid(pyramid):
    """Reconstruct a video from a laplacian video pyramid."""
    num_frames = pyramid[0].shape[0]
    for i in range(num_frames):
        frame_pyramid = [level[i] for level in pyramid]
        pyramid[0][i] = collapse_laplacian_pyramid(frame_pyramid)
    return pyramid[0]

## Motion magnification function:
Inputs: video data (YIQ float array), fps, frequency band, amplification factor (alpha), pyramid levels, lambda_c (cutoff spatial wavelength for adaptive amplification), and chrom_attenuation (color channel attenuation).

The function follows the reference MATLAB implementation (`amplify_spatial_lpyr_temporal_ideal.m`):
1. Build Laplacian video pyramid (spatial decomposition)
2. Ideal bandpass filter each pyramid level temporally
3. Amplify with adaptive per-level alpha based on lambda_c (Figure 6 of the paper)
4. Apply chromatic attenuation to I/Q channels
5. Add filtered signal back to pyramid and reconstruct

**Why level 0 and the coarsest level are skipped (zeroed out):**
- **Level 0 (finest, full resolution)** — captures the highest spatial frequencies (sharpest edges and fine details). The spatial wavelengths are so short that even modest amplification breaks the first-order Taylor approximation, producing ringing and ghosting artifacts.
- **Coarsest level (low-pass residual)** — this is not a true bandpass level; it is the Gaussian remainder (`gauss[-1]`) appended directly to the pyramid. It contains the DC component (overall mean intensity), so amplifying it would shift global brightness rather than reveal temporal variations.

In [ ]:
def eulerian_magnification(vid_data, fps, freq_min, freq_max,
                           amplification, pyramid_levels=4,
                           lambda_c=1000, chrom_attenuation=1.0):
    """Return the motion magnified video.

    Follows the reference MATLAB implementation with adaptive per-level
    amplification and chromatic attenuation.
    """
    height, width = vid_data.shape[1], vid_data.shape[2]
    alpha = amplification
    n_levels = pyramid_levels

    print("Creating video pyramid...")
    vid_pyramid = create_laplacian_video_pyramid(vid_data, n_levels)
    del vid_data

    print("Filtering and amplifying...")

    # Adaptive per-level amplification (matches MATLAB reference, Figure 6)
    delta = lambda_c / 8.0 / (1.0 + alpha)
    exaggeration_factor = 2.0

    # Representative wavelength for the coarsest level
    lambda_val = math.sqrt(height ** 2 + width ** 2) / 3.0

    # Compute per-level alpha from coarsest to finest
    level_alphas = [0.0] * n_levels
    lv = lambda_val
    for i in range(n_levels - 1, -1, -1):
        curr_alpha = (lv / delta / 8.0 - 1.0) * exaggeration_factor
        if i == n_levels - 1 or i == 0:
            # Level 0 (finest): spatial wavelengths too short, amplification
            # would break the Taylor approximation -> artifacts.
            # Coarsest level: low-pass residual (DC/mean), not a bandpass
            # level, amplifying it shifts global brightness.
            level_alphas[i] = 0.0
        elif curr_alpha > alpha:
            level_alphas[i] = alpha
        else:
            level_alphas[i] = curr_alpha
        lv /= 2.0

    # Filter, amplify, and add back
    for i in range(n_levels):
        if level_alphas[i] == 0.0:
            continue

        filtered = temporal_bandpass_filter(
            vid_pyramid[i], fps, freq_min=freq_min, freq_max=freq_max
        )
        # Amplify: full alpha on Y, attenuated on I/Q
        filtered[:, :, :, 0] *= level_alphas[i]
        filtered[:, :, :, 1] *= level_alphas[i] * chrom_attenuation
        filtered[:, :, :, 2] *= level_alphas[i] * chrom_attenuation

        vid_pyramid[i] += filtered
        del filtered

    print("Reconstructing from pyramid...")
    result = collapse_laplacian_video_pyramid(vid_pyramid)
    print("Done")
    return result

## Show Frequencies in the video 
We have also created a helper function that calculates the mean intensity variation in the input frames of the video. This helps us to figure out the cutoff frequencies required for our video.

In [ ]:
def show_frequencies(vid_data, fps, bounds=None):
    """Graph the average pixel value and frequency strength."""
    averages = []

    if bounds:
        for x in range(1, vid_data.shape[0] - 1):
            region = vid_data[
                x, bounds[2]:bounds[3], bounds[0]:bounds[1], :
            ]
            averages.append(region.sum())
    else:
        for x in range(1, vid_data.shape[0] - 1):
            averages.append(vid_data[x, :, :, :].sum())

    averages = averages - min(averages)

    charts_x = 1
    charts_y = 2
    pyplot.figure(figsize=(20, 10))
    pyplot.subplots_adjust(hspace=.7)

    pyplot.subplot(charts_y, charts_x, 1)
    pyplot.title("Pixel Average")
    pyplot.xlabel("Time")
    pyplot.ylabel("Brightness")
    pyplot.plot(averages)

    freqs = scipy.fftpack.fftfreq(len(averages), d=1.0 / fps)
    fft = abs(scipy.fftpack.fft(averages))
    idx = np.argsort(freqs)

    pyplot.subplot(charts_y, charts_x, 2)
    pyplot.title("FFT")
    pyplot.xlabel("Freq (Hz)")
    freqs = freqs[idx]
    fft = fft[idx]

    freqs = freqs[len(freqs) // 2 + 1:]
    fft = fft[len(fft) // 2 + 1:]
    pyplot.plot(freqs, abs(fft))

    pyplot.show()

## Downloading the input video to our notebook
To do a demo of our motion magnification algorithm we use a video which was used in the original paper. </br>
You can alternatively use you own video as input too. You will have to upload the video to the colab notebook, and rename the 'filename' variable as the name of your video.</br>
If you want to try your own video, comment the first line in the next code block and Uncomment the next line and enter your own filename.


In [ ]:
# If you want to try your own video, change the filename below
filename = "video.mp4"

##helper function for downloading video
You can skip the downloading part in this notebook if you are using your own video. 

In [ ]:
def download_file(url, dest_filename):
    """Download a file from the given url."""
    if os.path.isfile(dest_filename):
        print(f"Already Downloaded: {url} to {dest_filename}")
        return
    print(f"Downloading: {url} to {dest_filename}")

    response = requests.get(url, stream=True)
    if not response.ok:
        raise IOError(f"Couldn't download file: {url}")

    with open(dest_filename, 'wb') as fp:
        for block in response.iter_content(1024):
            fp.write(block)

## Downloading....
Skip the next code block if you are using your own video.

In [ ]:
download_file(
    'https://people.csail.mit.edu/mrub/evm/video/face.mp4',
    filename
)

## Final Demo on "input.mp4"
load the video and use show_frequencies to see the frequency range of the input video.

In [ ]:
vid, fps = load_video(filename)
show_frequencies(vid, fps)

You can use this information for making changes in the cutoff frequencies in the next code block if you want to.

In [ ]:
lower_hertz = 0.5
upper_hertz = 2
amplification_factor = 50
pyramid_levels = 4
lambda_c = 1000
chrom_attenuation = 1.0

Motion magnify the loaded video:

In [ ]:
mag_vid = eulerian_magnification(
    vid, fps,
    freq_min=lower_hertz,
    freq_max=upper_hertz,
    amplification=amplification_factor,
    pyramid_levels=pyramid_levels,
    lambda_c=lambda_c,
    chrom_attenuation=chrom_attenuation
)

save video to disk.

In [ ]:
out_name = (
    f"{os.path.splitext(filename)[0]}"
    f"_levels_{pyramid_levels}"
    f"_min_{lower_hertz}"
    f"_max_{upper_hertz}"
    f"_amp_{amplification_factor}"
    f"_lc_{lambda_c}"
    f"_chrom_{chrom_attenuation}.avi"
)

save_video(mag_vid, fps, out_name)

##Conclusion
To amplify motion, EVM does not perform feature
tracking or optical flow computation, but merely magnifies temporal color changes using spatio-temporal processing. This Eulerian based method, which temporally processes pixels in a fixed spatial
region, successfully reveals informative signals and amplifies small motions in real-world videos.

One drawback of this method is that we can see that we get artifacts in our videos as we increase amplification factor.


## If you like this content, follow me
<a href="https://twitter.com/joelk1jose" target="_blank"><img class="ai-subscribed-social-icon" src="https://github.com/joeljose/assets/blob/master/images/tw.png?raw=True" width="30"></a>
<a href="https://github.com/joeljose" target="_blank"><img class="ai-subscribed-social-icon" src="https://github.com/joeljose/assets/blob/master/images/gthb.png?raw=True" width="30"></a>
<a href="https://www.linkedin.com/in/joel-jose-527b80102/" target="_blank"><img class="ai-subscribed-social-icon" src="https://github.com/joeljose/assets/blob/master/images/lnkdn.png?raw=True" width="30"></a>

<h3 align="center">Show your support by starring the repository 🙂</h3>